# Fair model selection (Section 4) - working notebook

This notebook holds the cells for **Section 4: Select fairly** of the project guide.
They will be merged into `project.ipynb`. It has three parts:

| Part | Merge into project.ipynb? | Where |
|---|---|---|
| **A. Development setup** | **No** - project.ipynb already does this | - |
| **B. Shared evaluation helpers** | Yes | After preprocessing, *before* the baseline section, so the CNN code can use them |
| **C. Section 4: Model selection** | Yes | After the controlled CNN experiment, before the test evaluation |

Until both CNN runs exist, Part C runs with the baseline only and marks the
selection as **provisional**. The test set is never loaded in this notebook.

## Part A - Development setup (do not merge)

Re-creates the variables that `project.ipynb` defines in its data-preparation and
baseline sections (`DATA_DIR`, `load_image_split`, `X_val_images`, `y_val`,
`baseline_model`, `baseline_val_probabilities`, `baseline_training_seconds`), using the
same settings, so this notebook runs on its own.

The data folder is read from the `CASTING_DATA_DIR` environment variable if set;
otherwise `data/casting_512x512/` is used, then `data/`. Each must contain
`def_front/` and `ok_front/`.

In [ ]:
# DEV ONLY - project.ipynb already defines everything in this cell.
import os
import time
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
from sklearn.linear_model import LogisticRegression

SEED = 770
IMAGE_SIZE = (64, 64)


def find_data_dir():
    configured = os.environ.get("CASTING_DATA_DIR")
    options = (
        [Path(configured)] if configured
        else [Path("data") / "casting_512x512", Path("data")]
    )
    for option in options:
        if (option / "def_front").is_dir() and (option / "ok_front").is_dir():
            return option
    raise FileNotFoundError(
        "Could not find def_front/ and ok_front/ in: "
        + ", ".join(str(option) for option in options)
    )


DATA_DIR = find_data_dir()
manifest = pd.read_csv(Path("splits") / "split_manifest.csv")


def load_image_split(split_name):
    rows = (
        manifest[manifest["split"] == split_name]
        .sort_values("relative_path")
        .reset_index(drop=True)
    )
    images = []
    for relative_path in rows["relative_path"]:
        with Image.open(DATA_DIR / relative_path) as image:
            image = image.convert("L").resize(
                IMAGE_SIZE, resample=Image.Resampling.BILINEAR
            )
            images.append(np.asarray(image, dtype=np.float32) / 255.0)
    return np.stack(images), rows["label"].to_numpy(dtype=np.int64), rows


X_train_images, y_train, train_rows = load_image_split("train")
X_val_images, y_val, val_rows = load_image_split("validation")
X_train_flat = X_train_images.reshape(len(X_train_images), -1)
X_val_flat = X_val_images.reshape(len(X_val_images), -1)

baseline_model = LogisticRegression(
    solver="lbfgs", C=1.0, max_iter=2000, tol=1e-4, random_state=SEED,
)
start_time = time.perf_counter()
baseline_model.fit(X_train_flat, y_train)
baseline_training_seconds = time.perf_counter() - start_time

defect_column = list(baseline_model.classes_).index(1)
baseline_val_probabilities = baseline_model.predict_proba(X_val_flat)[:, defect_column]

print("Data folder:", DATA_DIR.as_posix())
print("Training images:", X_train_images.shape, "| Validation images:", X_val_images.shape)
print(f"Baseline fitted in {baseline_training_seconds:.2f} s "
      f"({int(baseline_model.n_iter_[0])} iterations)")

## Part B - Shared evaluation helpers

Every configuration is scored with the same code. The decision rule is fixed by the
project guide: **an image is defective when its predicted probability of defect is at
least 0.5.** This threshold is never tuned.

**How a CNN plugs in.** For each CNN run, the CNN section should:

1. During training, call `checkpoint_selector.update(epoch, model, val_probabilities)` once
   per epoch. It keeps the epoch with the highest **validation** defect F1. On ties it
   keeps the earliest epoch.
2. After training, call `checkpoint_selector.restore_best(model)`.
3. Register the run with `make_candidate(...)`, using `functools.partial` so that each
   candidate keeps its own model (see the example below).

```python
from functools import partial

def cnn_predict_proba(model, images):
    # images: NumPy array (N, 64, 64) in [0, 1] -> P(defective), shape (N,)
    model.eval()
    with torch.no_grad():
        batch = torch.as_tensor(images, dtype=torch.float32).unsqueeze(1).to(device)
        return torch.sigmoid(model(batch)).squeeze(1).cpu().numpy()
        # if the model outputs 2 logits instead: torch.softmax(model(batch), dim=1)[:, 1]

checkpoint_selector = ValidationCheckpointSelector(y_val)
for epoch in range(1, NUM_EPOCHS + 1):
    ...  # train one epoch
    checkpoint_selector.update(epoch, cnn_base_model,
                               cnn_predict_proba(cnn_base_model, X_val_images))
checkpoint_selector.restore_best(cnn_base_model)

cnn_base_candidate = make_candidate(
    name="CNN - base",
    val_probabilities=cnn_predict_proba(cnn_base_model, X_val_images),
    training_seconds=cnn_base_training_seconds,
    trainable_parameters=count_trainable_parameters(cnn_base_model),
    predict_proba=partial(cnn_predict_proba, cnn_base_model),
    selected_epoch=checkpoint_selector.best_epoch,
)
# Repeat with a new selector and the controlled change -> cnn_variant_candidate
```

In [ ]:
import copy
import hashlib

from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)

DECISION_THRESHOLD = 0.5  # fixed by the project guide; never tuned
DEFECTIVE = 1
ACCEPTABLE = 0


def predict_labels(defect_probabilities):
    """Apply the fixed rule: P(defective) >= 0.5 -> defective (1)."""
    probabilities = np.asarray(defect_probabilities, dtype=np.float64)
    return (probabilities >= DECISION_THRESHOLD).astype(np.int64)


def defect_metrics(y_true, defect_probabilities):
    """Defective-class precision, recall, F1, plus accuracy and error counts."""
    y_pred = predict_labels(defect_probabilities)
    tn, fp, fn, tp = confusion_matrix(
        y_true, y_pred, labels=[ACCEPTABLE, DEFECTIVE]
    ).ravel()
    return {
        "Precision": precision_score(y_true, y_pred, pos_label=DEFECTIVE, zero_division=0),
        "Recall": recall_score(y_true, y_pred, pos_label=DEFECTIVE, zero_division=0),
        "F1": f1_score(y_true, y_pred, pos_label=DEFECTIVE, zero_division=0),
        "Accuracy": accuracy_score(y_true, y_pred),
        "False negatives": int(fn),  # defects accepted
        "False positives": int(fp),  # acceptable parts rejected
    }


def count_trainable_parameters(model):
    """Trainable parameters of a PyTorch model or a fitted sklearn linear model."""
    if hasattr(model, "parameters"):
        return int(sum(p.numel() for p in model.parameters() if p.requires_grad))
    return int(model.coef_.size + model.intercept_.size)


def check_probabilities(name, probabilities, expected_length):
    probabilities = np.asarray(probabilities, dtype=np.float64).ravel()
    if probabilities.shape != (expected_length,):
        raise ValueError(f"{name}: expected {expected_length} probabilities, "
                         f"got shape {probabilities.shape}.")
    if not np.isfinite(probabilities).all() or probabilities.min() < 0 or probabilities.max() > 1:
        raise ValueError(f"{name}: values must be probabilities of defect in [0, 1] "
                         "(apply sigmoid/softmax to logits first).")
    return probabilities


def make_candidate(name, val_probabilities, training_seconds,
                   trainable_parameters, predict_proba, selected_epoch=None):
    """Package one fitted configuration for validation-based selection.

    predict_proba must map a NumPy array of prepared images, shape (N, 64, 64),
    to P(defective), shape (N,), using the frozen model or checkpoint.
    """
    return {
        "name": name,
        "val_probabilities": check_probabilities(name, val_probabilities, len(y_val)),
        "training_seconds": float(training_seconds),
        "trainable_parameters": int(trainable_parameters),
        "predict_proba": predict_proba,
        "selected_epoch": selected_epoch,
    }


class ValidationCheckpointSelector:
    """Keeps the CNN epoch with the highest validation defect F1.

    Ties keep the earliest epoch (less training). Only validation data is used.
    The best weights are copied in memory; nothing is written to disk.
    """

    def __init__(self, y_true):
        self.y_true = np.asarray(y_true)
        self.best_epoch = None
        self.best_f1 = -np.inf
        self.best_state = None
        self.history = []

    def update(self, epoch, model, val_probabilities):
        probabilities = check_probabilities(
            f"epoch {epoch}", val_probabilities, len(self.y_true)
        )
        metrics = defect_metrics(self.y_true, probabilities)
        self.history.append({"Epoch": epoch, **metrics})
        if metrics["F1"] > self.best_f1:  # strict: ties keep the earlier epoch
            self.best_f1 = metrics["F1"]
            self.best_epoch = epoch
            self.best_state = copy.deepcopy(model.state_dict())
        return metrics

    def restore_best(self, model):
        if self.best_state is None:
            raise RuntimeError("No epoch has been recorded.")
        model.load_state_dict(self.best_state)
        model.eval()
        print(f"Restored epoch {self.best_epoch} "
              f"(validation defect F1 = {self.best_f1:.4f}).")
        return model

    def history_table(self):
        return pd.DataFrame(self.history).set_index("Epoch")

## Part C - Section 4: Model selection on validation data

**Selection rule, fixed before any results were seen:**

1. **Metric:** F1 for the defective class (label 1) on the **validation** set.
2. **Decision rule:** probability of defect ≥ 0.5 means defective. This is the same for
   every model and never changed.
3. **Ties go to the simpler model.** Logistic regression (a single linear layer with
   4,097 parameters) is always treated as the simplest. Between CNNs, the one with fewer
   trainable parameters is simpler. If two CNNs still tie (for example, the controlled
   change was dropout, which adds no parameters), the base CNN is preferred over the
   controlled variant.
4. **CNN checkpoints** come from validation F1 only (see `ValidationCheckpointSelector`).
5. **No test data** is loaded or scored until this selection is final. The selected
   model is then frozen and evaluated on the test set once.

Before scoring, each candidate's stored validation probabilities are recomputed from its
frozen predictor. This confirms that the model being selected is the one that will be
tested, for example that the CNN's restored checkpoint was not overwritten by later
training.

In [ ]:
# Register the three configurations, in order: baseline, base CNN, controlled variant.
baseline_candidate = make_candidate(
    name="Logistic regression",
    val_probabilities=baseline_val_probabilities,
    training_seconds=baseline_training_seconds,
    trainable_parameters=count_trainable_parameters(baseline_model),
    predict_proba=lambda images: baseline_model.predict_proba(
        np.asarray(images).reshape(len(images), -1)
    )[:, list(baseline_model.classes_).index(DEFECTIVE)],
)

candidates = [baseline_candidate]
missing_candidates = []
for variable_name in ["cnn_base_candidate", "cnn_variant_candidate"]:
    if variable_name in globals():
        candidates.append(globals()[variable_name])
    else:
        missing_candidates.append(variable_name)

# Confirm each frozen predictor reproduces its stored validation probabilities.
for candidate in candidates:
    recomputed = check_probabilities(
        candidate["name"], candidate["predict_proba"](X_val_images), len(y_val)
    )
    if not np.allclose(recomputed, candidate["val_probabilities"], atol=1e-5):
        raise AssertionError(
            f"{candidate['name']}: predict_proba does not reproduce the stored validation "
            "probabilities. The model changed after it was registered."
        )

print("Configurations registered:", [c["name"] for c in candidates])
if missing_candidates:
    print("Not yet available:", missing_candidates)

In [ ]:
comparison_rows = []
for order, candidate in enumerate(candidates):
    comparison_rows.append({
        "Configuration": candidate["name"],
        **defect_metrics(y_val, candidate["val_probabilities"]),
        "Training time (s)": candidate["training_seconds"],
        "Trainable parameters": candidate["trainable_parameters"],
        "Selected epoch": candidate["selected_epoch"],
        "Order": order,
    })
validation_comparison = pd.DataFrame(comparison_rows)

# Highest defect F1. Ties: logistic regression (Order 0) first, then fewer
# trainable parameters, then the base CNN before the controlled variant.
best_f1 = validation_comparison["F1"].max()
tied = validation_comparison[
    np.isclose(validation_comparison["F1"], best_f1, rtol=0, atol=1e-12)
].assign(is_cnn=lambda table: (table["Order"] != 0).astype(int))
winner = tied.sort_values(["is_cnn", "Trainable parameters", "Order"]).iloc[0]
selected_candidate = candidates[int(winner["Order"])]

validation_comparison["Selected"] = np.where(
    validation_comparison["Order"] == winner["Order"], "✓", ""
)

display(
    validation_comparison
    .drop(columns="Order")
    .set_index("Configuration")
    .style.format({
        "Precision": "{:.4f}", "Recall": "{:.4f}", "F1": "{:.4f}",
        "Accuracy": "{:.4f}", "Training time (s)": "{:.2f}",
        "Trainable parameters": "{:,}", "Selected epoch": "{:.0f}",
    }, na_rep="-")
    .set_caption(
        f"Validation set (n = {len(y_val)}; {int((y_val == DEFECTIVE).sum())} defective, "
        f"{int((y_val == ACCEPTABLE).sum())} acceptable). "
        f"Decision rule: P(defective) ≥ {DECISION_THRESHOLD}."
    )
)

print(f"Highest validation defect F1: {best_f1:.4f}")
if len(tied) > 1:
    print("Tied configurations:", ", ".join(tied["Configuration"]))
    print("Tie broken in favor of the simpler model "
          "(logistic regression, then fewer CNN parameters, then the base CNN).")
print("Selected configuration:", selected_candidate["name"])

In [ ]:
# Freeze the selection. The test-evaluation section must use selected_predict_proba
# and must call assert_selection_unchanged() before scoring the test set.
SELECTION_IS_FINAL = not missing_candidates
selected_name = selected_candidate["name"]
selected_predict_proba = selected_candidate["predict_proba"]
_selected_fingerprint = hashlib.sha256(
    np.round(selected_candidate["val_probabilities"], 6).tobytes()
).hexdigest()


def assert_selection_unchanged():
    """Fail if the selection is provisional or the frozen model has changed."""
    if not SELECTION_IS_FINAL:
        raise RuntimeError(
            "Selection is provisional (missing: " + ", ".join(missing_candidates) + "). "
            "Do not evaluate the test set yet."
        )
    current = check_probabilities(selected_name, selected_predict_proba(X_val_images), len(y_val))
    current_fingerprint = hashlib.sha256(np.round(current, 6).tobytes()).hexdigest()
    if current_fingerprint != _selected_fingerprint:
        raise RuntimeError(f"{selected_name} has changed since it was selected.")


print("Frozen selection:", selected_name)
if selected_candidate["selected_epoch"] is not None:
    print("Checkpoint epoch (chosen on validation F1):", selected_candidate["selected_epoch"])
if SELECTION_IS_FINAL:
    print("Selection is final. The test set may now be evaluated once.")
else:
    print("PROVISIONAL: waiting for", ", ".join(missing_candidates),
          "- do not evaluate the test set.")

### Selection result

*To be written after all three configurations have run.* State which configuration had the
highest validation defect F1, whether a tie-break was needed, and (for a CNN) which epoch
was kept. Note how close the runner-up was. With 117 defective validation images, a
difference of one or two predictions changes F1 by roughly 0.005-0.01.